# Daily Challenge — Build a Retrieval Augmented Generation (RAG) System
**Week 14 · Day 6** — LangChain + Hugging Face

<a href="https://colab.research.google.com/github/Raphzaf/DI-Bootcamp-Stage2/blob/main/Week14/Day6/DailyChallenge/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Goal: answer questions using the `databricks/databricks-dolly-15k` dataset as knowledge base.

Pipeline:
1. **Load** the dataset as LangChain `Document`s (`HuggingFaceDatasetLoader`)
2. **Split** documents into overlapping chunks (`RecursiveCharacterTextSplitter`)
3. **Embed** the chunks (`sentence-transformers/all-MiniLM-l6-v2` via `HuggingFaceEmbeddings`)
4. **Index** them in a `FAISS` vector store
5. **Retrieve** the top-k chunks for a question and let an **LLM** answer (`RetrievalQA`)

> 💡 In Colab: *Runtime → Change runtime type → T4 GPU* speeds up the embedding step (it also works on CPU, just slower).

## 0. Hugging Face warm-up — BERT embeddings

In [ ]:
!pip install -q transformers

In [ ]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

inputs = tokenizer("Hello Hugging Face!", return_tensors="pt")
outputs = model(**inputs)

print("Tokens:", tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))
print("last_hidden_state shape:", outputs.last_hidden_state.shape)  # (batch, n_tokens, 768)

`outputs.last_hidden_state` holds one 768-dimensional vector per token. These embeddings are what we will use (via a smaller sentence model) to find relevant documents in the RAG system.

## 1. Set up the environment

In [ ]:
!pip install -q langchain torch transformers sentence-transformers faiss-cpu
!pip install -q -U langchain-community langchain-huggingface langchain-text-splitters langchain-classic
!pip install -Uq datasets

In [ ]:
# Imports — the "new" packages first, with fallbacks to the legacy paths used in the exercise statement
try:
    from langchain_community.document_loaders import HuggingFaceDatasetLoader
except ImportError:
    from langchain.document_loaders import HuggingFaceDatasetLoader

try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter

try:
    from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings
    from langchain_community.llms import HuggingFacePipeline

try:
    from langchain_community.vectorstores import FAISS
except ImportError:
    from langchain.vectorstores import FAISS

try:
    from langchain_classic.chains import RetrievalQA      # LangChain >= 1.0
except ImportError:
    from langchain.chains import RetrievalQA              # LangChain 0.x

from langchain_core.prompts import PromptTemplate
from langchain_core.language_models.llms import LLM

import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## 2. Load the dataset

In [ ]:
dataset_name = "databricks/databricks-dolly-15k"
page_content_column = "context"

loader = HuggingFaceDatasetLoader(dataset_name, page_content_column)
data = loader.load()

print(len(data), "documents loaded")
print(data[:2])

Many Dolly rows (brainstorming, creative writing…) have an **empty** `context`. Empty documents carry no knowledge, so we drop them before indexing.

In [ ]:
data = [doc for doc in data if doc.page_content and doc.page_content.strip()]
print(len(data), "documents with a non-empty context")
print(data[0].page_content[:300])
print(data[0].metadata)

## 3. Split the documents

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
docs = text_splitter.split_documents(data)

print(len(docs), "chunks")
print(docs[0])

## 4. Embed the text

In [ ]:
modelPath = "sentence-transformers/all-MiniLM-l6-v2"
model_kwargs = {"device": device}          # the exercise uses 'cpu'; GPU is used automatically if available
encode_kwargs = {"normalize_embeddings": False}

embeddings = HuggingFaceEmbeddings(
    model_name=modelPath,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)

text = "This is a test document."
query_result = embeddings.embed_query(text)
print(len(query_result), "dimensions")
print(query_result[:3])

## 5. Create the FAISS vector store
*This step can take a few minutes on CPU (~1 min on a T4 GPU).*

In [ ]:
%%time
db = FAISS.from_documents(docs, embeddings)
print(db.index.ntotal, "vectors indexed")

In [ ]:
# Quick sanity check: pure similarity search, no LLM yet
question = "What is cheesemaking?"
for i, d in enumerate(db.similarity_search(question, k=3), 1):
    print(f"--- #{i} ---")
    print(d.page_content[:300], "...\n")

## 6. Prepare the LLM model

In [ ]:
model_name = "Intel/dynamic_tinybert"

tokenizer = AutoTokenizer.from_pretrained(model_name, padding=True, truncation=True, max_length=512)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

question_answerer = pipeline(
    "question-answering",
    model=model,
    tokenizer=tokenizer,
    device=0 if device == "cuda" else -1,
)

### ⚠️ Why `HuggingFacePipeline(pipeline=question_answerer)` does not work as-is

`HuggingFacePipeline` only accepts *generative* tasks (`text-generation`, `text2text-generation`, `summarization`, `translation`).
`Intel/dynamic_tinybert` is an **extractive** QA model: it needs a `question` **and** a `context` separately and returns a span of the context — it cannot take a single prompt string.

In [ ]:
try:
    llm = HuggingFacePipeline(
        pipeline=question_answerer,
        model_kwargs={"temperature": 0.7, "max_length": 512},
    )
    print(llm.invoke("What is cheesemaking?"))
except Exception as e:
    print(f"{type(e).__name__}: {e}")

**Solution:** wrap the QA pipeline in a small custom LangChain `LLM`.
We give the chain a prompt with clear `CONTEXT:` / `QUESTION:` markers; the wrapper splits the prompt back into context + question and calls the extractive model.

In [ ]:
from typing import Any, List, Optional

CONTEXT_TAG, QUESTION_TAG = "CONTEXT:", "QUESTION:"


class ExtractiveQALLM(LLM):
    """LangChain LLM wrapper around a Hugging Face `question-answering` pipeline."""

    qa_pipeline: Any
    max_answer_len: int = 60

    @property
    def _llm_type(self) -> str:
        return "hf_extractive_qa"

    def _call(self, prompt: str, stop: Optional[List[str]] = None, run_manager=None, **kwargs) -> str:
        if QUESTION_TAG in prompt:
            context, question = prompt.rsplit(QUESTION_TAG, 1)
            context = context.replace(CONTEXT_TAG, "", 1)
        else:  # no context given: answer from the prompt itself
            context, question = prompt, prompt
        result = self.qa_pipeline(
            question=question.strip(),
            context=context.strip(),
            max_answer_len=self.max_answer_len,
            truncation="only_second",
            max_seq_len=512,
            doc_stride=128,
        )
        return result["answer"].strip()


llm = ExtractiveQALLM(qa_pipeline=question_answerer)

# Test the wrapper directly
print(llm.invoke(f"{CONTEXT_TAG} Paris is the capital and largest city of France.\n{QUESTION_TAG} What is the capital of France?"))

## 7. Build the Retrieval QA Chain

In [ ]:
retriever = db.as_retriever(search_kwargs={"k": 4})

qa_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=f"{CONTEXT_TAG}\n{{context}}\n{QUESTION_TAG} {{question}}",
)

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",          # all retrieved chunks are "stuffed" into one context for the extractive model
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": qa_prompt},
)

> The statement suggests `chain_type="refine"`. *Refine* asks the LLM to **rewrite** a previous answer with each new chunk — that only makes sense for a generative model. With an extractive model, `"stuff"` is the right choice. The `refine` chain is shown in the bonus section below with a generative model.

## 8. Test the RAG system

In [ ]:
question = "What is cheesemaking?"

result = qa.invoke({"query": question})   # qa.run(...) is deprecated in recent LangChain versions
print("Question:", question)
print("Answer  :", result["result"])
print("\nSources:")
for d in result["source_documents"]:
    print(" -", d.page_content[:120].replace("\n", " "), "...")

In [ ]:
questions = [
    "What is cheesemaking?",
    "Who was the first person to walk on the moon?",
    "What is the capital of Australia?",
    "What is a dulcimer?",
]
for q in questions:
    print(f"Q: {q}\nA: {qa.invoke({'query': q})['result']}\n")

## 9. Bonus — `refine` chain with a generative model

To use `chain_type="refine"` exactly as in the statement, we need a model that *generates* text. `google/flan-t5-base` (`text2text-generation`) works directly with `HuggingFacePipeline`.

In [ ]:
gen_pipe = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    max_new_tokens=128,
    device=0 if device == "cuda" else -1,
)
gen_llm = HuggingFacePipeline(pipeline=gen_pipe)

qa_refine = RetrievalQA.from_chain_type(
    llm=gen_llm,
    chain_type="refine",
    retriever=retriever,
    return_source_documents=False,
)

question = "What is cheesemaking?"
print(qa_refine.invoke({"query": question})["result"])

## Conclusion

- **Retrieval** (MiniLM embeddings + FAISS) finds the Dolly passages relevant to the question in milliseconds.
- **Generation / extraction**: the LLM answers *from these passages* instead of from its own memory — this is the core idea of RAG (fewer hallucinations, answers grounded in a knowledge base that can be updated without retraining).
- `Intel/dynamic_tinybert` is extractive: answers are short spans copied from the retrieved context. A generative model (Flan-T5, or a larger instruction LLM) produces fuller sentences and supports the `refine` chain.
- Quality depends heavily on the knowledge base: Dolly only covers some topics, so questions outside it get weaker answers.